# GISRUK2025 Poster Figures
This file is originally located at the root folder. It is modified from whitelee2.ipynb for making figures for GISRUK2025 poster.

In [1]:
# import useful libraries

import numpy as np
import netCDF4
import matplotlib.pyplot as plt
import folium.features
import folium.raster_layers
import folium
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

from Land.main import land, feasibility
from CRS.crs_init import CRSConvertor
from Optimiser.config_ss import *
from Optimiser.main import *

ModuleNotFoundError: No module named 'Land'

# 1. Current Layout
First, we display the original wind farm layout here.

In [ ]:
# prerequisites
# read wind turbine locations
turbine_loc = np.loadtxt('Aborted/case/Turbines_At_Whitelee_Wind_Farm.csv', delimiter=',', encoding='utf-8', usecols=[1, 2])

image_file = image_file = 'https://github.com/ShitianZhang22/Wind-Farm-Layout-Optimisation/blob/main/icon/turbine.png?raw=true'

_wind_data = np.array([[2.3046875, 0.0390625],
                       [3.1015625, 0.125    ],
                       [2.8203125, 0.09375  ],
                       [2.5      , 0.078125 ],
                       [3.9375   , 0.140625 ],
                       [4.2890625, 0.2265625],
                       [3.9296875, 0.2265625],
                       [2.9140625, 0.078125 ]])

In [ ]:
# Create a folium map centered around the site
m = folium.Map(location=[55.6741, -4.2713], zoom_start=12, tiles='CartoDB.PositronNoLabels')

# Visualize the boundary on a folium map
boundary_file = 'Land/data/WhiteleeBoundary.geojson'
boundary = gpd.read_file(boundary_file)

# Calculate the bounding box
bounds = boundary.total_bounds  # [minx, miny, maxx, maxy]
bounding_box = [[bounds[1], bounds[0]], [bounds[3], bounds[2]]]

# Add the bounding box to the map. This part is currently commented out but may be useful in the future.
# folium.Rectangle(
#     bounds=bounding_box,
#     color='blue',
#     weight=2,
#     fill=False
# ).add_to(m)

# Remove the filling display of the geojson file but keep the boundary
folium.GeoJson(
    boundary,
    style_function=lambda x: {
        'fillOpacity': 0.1,  # Make the fill transparent
        'fillColor': '#FF0000',
        'color': '#500000',  # Keep the boundary visible
        'weight': 2
    }
).add_to(m)

# Add the current wind turbines to the map
for i in range(turbine_loc.shape[0]):  # add turbine locations
    folium.Marker(
        location=[turbine_loc[i, 0], turbine_loc[i, 1]],
        icon=folium.CustomIcon(icon_image=image_file, icon_size=(20, 20), icon_anchor=(10, 20)),
        color='red'
    ).add_to(m)

# Display land cover type within the geojson boundary
land_cover_file = 'Land/raw/C3S-LC-L4-LCCS-Map-300m-P1Y-2022-v2.1.1.nc'
with netCDF4.Dataset(land_cover_file, 'r') as file:
    land_class = file.variables['lccs_class']
    lat = file.variables['lat'][:]
    lon = file.variables['lon'][:]

    # Filter points within the bounding box
    y_range = np.argwhere((lat < bounds[3]) & (lat > bounds[1])).flatten()
    x_range = np.argwhere((lon < bounds[2]) & (lon > bounds[0])).flatten()

    # Extract the land cover data within the bounding box
    land_cover = land_class[0, y_range.min():y_range.max()+1, x_range.min():x_range.max()+1].data

    # Create an RGBA image for the land cover data
    rgba_img = np.zeros((len(y_range), len(x_range), 4), dtype=np.uint8)
    colourmap = {
        70: [0, 60, 0, 128],
        100: [140, 160, 0, 128],
        110: [190, 150, 0, 128],
        130: [255, 180, 50, 128],
        180: [0, 220, 130, 128],
        210: [0, 70, 200, 128],
    }

    for key, color in colourmap.items():
        rgba_img[land_cover == key] = color

    # Assign transparent color to data outside the geojson boundary
    for i, lat_val in enumerate(lat[y_range]):
        for j, lon_val in enumerate(lon[x_range]):
            point = Point(lon_val, lat_val)
            if not boundary.union_all().contains(point):
                rgba_img[i, j, 3] = 0  # Set alpha to 0 for transparency

    # Add the RGBA image to the map
    # folium.raster_layers.ImageOverlay(
    #     image=rgba_img,
    #     bounds=bounding_box,
    #     origin='upper',
    #     opacity=1
    # ).add_to(m)

# Add a legend to the map
legend_html = '''
    <div style="position: fixed; 
                bottom: 50px; right: 50px; width: auto; height: auto; 
                background-color: rgba(255, 255, 255, 0.8); border: 2px solid black; z-index:9999; font-size:14px;">
        <h4 style="margin: 10px; font-weight: bold; font-size: 20px;">Legend</h4>
        <ul style="list-style: none; padding: 0; margin: 10px;">
            <li><img src="https://github.com/ShitianZhang22/Wind-Farm-Layout-Optimisation/blob/main/icon/turbine-small.png?raw=true" style="width: 15px; height: 15px; margin-right: 2px;">Wind Turbine</li>
            <li><span style="background-color: black; width: 12px; height: 3px; display: inline-block; margin-right: 5px; margin-bottom: 4px;"></span>Site Boundary</li>
            <li><span style="background-color: rgba(0, 60, 0, 0.5); width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></span>Tree Cover, Needleleaved, Evergreen, Closed to Open</li>
            <li><span style="background-color: rgba(140, 160, 0, 0.5); width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></span>Mosaic Tree and Shrub (>50%) / Herbaceous Cover (<50%)</li>
            <li><span style="background-color: rgba(190, 150, 0, 0.5); width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></span>Mosaic Herbaceous Cover (>50%) / Tree and Shrub (<50%)</li>
            <li><span style="background-color: rgba(255, 180, 50, 0.5); width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></span>Grassland</li>
            <li><span style="background-color: rgba(0, 220, 130, 0.5); width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></span>Shrub or Herbaceous Cover, Flooded, Fresh/Saline/Brackish Water</li>
            <li><span style="background-color: rgba(0, 70, 200, 0.5); width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></span>Water Bodies</li>
        </ul>
    </div>
'''
# m.get_root().html.add_child(folium.Element(legend_html))

# Display the map
m

# 2. Optimisation

Here we do some optimisation. This part can be skipped if the first line is not commented.

First, we need the feasible cells.

In [ ]:
# Here are some prerequisites.
# this is for converting the CRS
print('Bounding box:')
print(bounds, '\n')

conv = CRSConvertor([bounds[3], bounds[0], bounds[1], bounds[2]], cell_width)
print('Grid size:')
print(conv.rows, conv.cols)

rows = conv.rows
cols = conv.cols

Bounding box:
[-4.40025458 55.63733847 -4.15341513 55.72078914] 

Grid size:
58 101


In [ ]:
%%script true
# The result is at the next cell.
# First, we need to filter the cells within the real boundary from the geojson file.
within_boundary = np.zeros((conv.rows * conv.cols), dtype=bool)

for i in range(conv.grid_gcs.shape[0]):
    point = Point(conv.grid_gcs[i, 1], conv.grid_gcs[i, 0])
    if boundary.union_all().contains(point):
        within_boundary[i] = True
print('The number of cells within the boundary:', np.sum(within_boundary))

# Then, we check the feasibility of the grid cells.
feasible_ind = land('Land/data/infeasible.nc', conv.grid_gcs)
# A stupid step: convert the feasible_ind back to boolean
feasible_temp = np.zeros((conv.rows * conv.cols), dtype=bool)
for i in range(len(feasible_ind)):
    feasible_temp[feasible_ind[i]] = True
print('The number of feasible cells:', np.sum(feasible_temp))

gene_space = feasible_temp & within_boundary
print('The final feasible cells within the boundary:', np.sum(gene_space))
gene_space = np.argwhere(gene_space).T[0].tolist()
print('The feasible cells are:', gene_space)

In [ ]:
## feasible cells
gene_space = [19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 221, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 350, 351, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 447, 448, 449, 450, 451, 452, 534, 535, 536, 537, 538, 539, 540, 541, 542, 547, 548, 549, 550, 551, 552, 553, 637, 638, 639, 640, 641, 642, 643, 648, 649, 650, 651, 652, 653, 654, 655, 669, 736, 737, 738, 739, 740, 741, 742, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 770, 771, 772, 773, 774, 777, 836, 837, 838, 839, 840, 841, 842, 843, 844, 845, 846, 847, 848, 849, 850, 851, 852, 853, 854, 855, 856, 857, 858, 859, 860, 861, 862, 869, 870, 872, 873, 874, 875, 876, 877, 878, 879, 880, 937, 938, 939, 940, 941, 942, 943, 944, 945, 946, 947, 948, 949, 950, 951, 952, 953, 954, 955, 956, 957, 958, 959, 960, 961, 962, 963, 964, 969, 970, 971, 972, 973, 974, 975, 976, 977, 978, 979, 980, 981, 1037, 1038, 1039, 1040, 1041, 1042, 1043, 1044, 1045, 1046, 1047, 1048, 1049, 1050, 1051, 1052, 1053, 1054, 1055, 1056, 1057, 1058, 1059, 1060, 1061, 1062, 1063, 1064, 1065, 1069, 1070, 1071, 1072, 1073, 1074, 1075, 1076, 1077, 1078, 1079, 1080, 1081, 1082, 1139, 1140, 1141, 1142, 1143, 1144, 1145, 1146, 1147, 1148, 1149, 1150, 1151, 1152, 1153, 1154, 1155, 1156, 1157, 1158, 1159, 1160, 1161, 1162, 1163, 1164, 1165, 1166, 1170, 1171, 1172, 1173, 1174, 1175, 1176, 1177, 1178, 1179, 1180, 1181, 1182, 1240, 1241, 1242, 1243, 1244, 1245, 1246, 1247, 1248, 1249, 1250, 1251, 1252, 1253, 1254, 1255, 1256, 1257, 1258, 1259, 1260, 1261, 1262, 1263, 1264, 1265, 1266, 1267, 1271, 1272, 1273, 1274, 1275, 1276, 1277, 1278, 1279, 1280, 1281, 1282, 1283, 1341, 1342, 1343, 1344, 1345, 1346, 1347, 1348, 1349, 1350, 1351, 1352, 1353, 1354, 1355, 1356, 1357, 1358, 1359, 1360, 1361, 1362, 1363, 1364, 1365, 1366, 1367, 1368, 1369, 1372, 1373, 1374, 1375, 1376, 1377, 1378, 1379, 1380, 1381, 1382, 1383, 1443, 1444, 1445, 1446, 1447, 1448, 1449, 1450, 1451, 1452, 1453, 1454, 1455, 1456, 1457, 1458, 1459, 1460, 1461, 1462, 1463, 1464, 1465, 1466, 1467, 1468, 1469, 1470, 1473, 1474, 1475, 1476, 1477, 1478, 1479, 1480, 1481, 1482, 1483, 1484, 1485, 1544, 1545, 1546, 1547, 1548, 1549, 1550, 1551, 1552, 1553, 1554, 1555, 1556, 1557, 1558, 1559, 1560, 1561, 1562, 1563, 1564, 1565, 1566, 1567, 1568, 1569, 1570, 1575, 1576, 1577, 1578, 1579, 1580, 1581, 1582, 1583, 1584, 1585, 1586, 1587, 1588, 1589, 1646, 1647, 1648, 1649, 1650, 1651, 1652, 1653, 1654, 1655, 1656, 1657, 1658, 1659, 1660, 1661, 1662, 1663, 1664, 1665, 1666, 1667, 1668, 1669, 1670, 1671, 1676, 1677, 1678, 1679, 1680, 1681, 1682, 1683, 1684, 1685, 1686, 1687, 1688, 1689, 1690, 1691, 1692, 1748, 1749, 1750, 1751, 1752, 1753, 1754, 1755, 1756, 1757, 1758, 1759, 1760, 1761, 1762, 1763, 1764, 1765, 1766, 1767, 1768, 1769, 1770, 1771, 1772, 1777, 1778, 1779, 1780, 1781, 1782, 1783, 1784, 1785, 1786, 1787, 1788, 1789, 1790, 1791, 1792, 1793, 1842, 1844, 1845, 1850, 1851, 1852, 1853, 1854, 1855, 1856, 1857, 1858, 1859, 1860, 1861, 1862, 1863, 1864, 1865, 1866, 1867, 1868, 1869, 1870, 1871, 1872, 1876, 1877, 1878, 1879, 1880, 1881, 1882, 1883, 1884, 1885, 1886, 1887, 1888, 1889, 1890, 1891, 1892, 1893, 1894, 1943, 1944, 1945, 1946, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 2042, 2043, 2044, 2045, 2046, 2047, 2048, 2050, 2051, 2052, 2053, 2054, 2055, 2056, 2057, 2058, 2059, 2060, 2061, 2062, 2063, 2064, 2065, 2066, 2067, 2068, 2069, 2070, 2071, 2072, 2073, 2074, 2075, 2076, 2077, 2078, 2079, 2080, 2081, 2082, 2083, 2084, 2085, 2086, 2087, 2088, 2089, 2090, 2091, 2092, 2093, 2094, 2095, 2096, 2097, 2142, 2143, 2144, 2145, 2146, 2147, 2148, 2149, 2150, 2151, 2152, 2153, 2154, 2155, 2156, 2157, 2158, 2159, 2160, 2161, 2162, 2163, 2164, 2165, 2166, 2167, 2168, 2169, 2170, 2171, 2172, 2173, 2174, 2175, 2176, 2177, 2178, 2179, 2180, 2181, 2182, 2183, 2184, 2185, 2186, 2187, 2188, 2189, 2190, 2191, 2192, 2193, 2194, 2195, 2196, 2197, 2198, 2199, 2203, 2204, 2208, 2209, 2244, 2245, 2246, 2247, 2248, 2249, 2250, 2251, 2252, 2253, 2254, 2255, 2256, 2257, 2258, 2259, 2260, 2261, 2262, 2263, 2264, 2265, 2266, 2267, 2268, 2269, 2270, 2271, 2272, 2273, 2274, 2275, 2276, 2277, 2278, 2279, 2280, 2281, 2282, 2283, 2284, 2285, 2286, 2287, 2288, 2289, 2290, 2291, 2292, 2293, 2294, 2295, 2296, 2297, 2298, 2299, 2300, 2303, 2304, 2305, 2306, 2309, 2310, 2311, 2342, 2343, 2344, 2345, 2346, 2347, 2348, 2349, 2350, 2351, 2352, 2353, 2354, 2355, 2356, 2357, 2358, 2359, 2360, 2361, 2362, 2363, 2364, 2365, 2366, 2367, 2368, 2369, 2370, 2371, 2372, 2373, 2374, 2375, 2376, 2377, 2378, 2379, 2380, 2381, 2382, 2383, 2384, 2385, 2386, 2387, 2388, 2389, 2390, 2391, 2392, 2393, 2394, 2395, 2396, 2397, 2398, 2399, 2400, 2403, 2404, 2405, 2406, 2407, 2408, 2409, 2410, 2411, 2412, 2413, 2414, 2443, 2444, 2445, 2446, 2447, 2448, 2449, 2450, 2451, 2452, 2453, 2454, 2455, 2456, 2457, 2458, 2459, 2460, 2461, 2462, 2463, 2464, 2465, 2466, 2467, 2468, 2469, 2470, 2471, 2472, 2473, 2474, 2475, 2476, 2477, 2478, 2479, 2480, 2481, 2482, 2483, 2484, 2485, 2486, 2487, 2488, 2489, 2490, 2491, 2492, 2493, 2494, 2495, 2496, 2497, 2498, 2499, 2500, 2501, 2504, 2505, 2506, 2507, 2508, 2509, 2510, 2511, 2512, 2513, 2514, 2515, 2516, 2517, 2520, 2521, 2543, 2544, 2545, 2546, 2547, 2548, 2549, 2550, 2551, 2552, 2553, 2554, 2555, 2556, 2557, 2558, 2559, 2560, 2561, 2562, 2563, 2564, 2565, 2566, 2567, 2568, 2569, 2570, 2571, 2572, 2573, 2574, 2575, 2576, 2577, 2578, 2579, 2580, 2581, 2582, 2583, 2584, 2585, 2586, 2587, 2588, 2589, 2590, 2591, 2592, 2593, 2594, 2595, 2596, 2597, 2598, 2599, 2600, 2601, 2605, 2606, 2607, 2608, 2609, 2610, 2611, 2612, 2613, 2614, 2615, 2616, 2617, 2618, 2619, 2620, 2621, 2622, 2623, 2624, 2625, 2634, 2635, 2636, 2637, 2643, 2645, 2646, 2647, 2648, 2649, 2650, 2651, 2652, 2653, 2654, 2655, 2656, 2657, 2658, 2659, 2660, 2661, 2662, 2663, 2664, 2665, 2666, 2667, 2668, 2669, 2670, 2671, 2672, 2673, 2674, 2675, 2676, 2677, 2678, 2679, 2680, 2681, 2682, 2683, 2684, 2685, 2686, 2687, 2688, 2689, 2690, 2691, 2692, 2693, 2694, 2695, 2696, 2697, 2698, 2699, 2700, 2701, 2704, 2705, 2706, 2707, 2708, 2709, 2710, 2711, 2712, 2713, 2714, 2715, 2716, 2717, 2718, 2719, 2720, 2721, 2722, 2723, 2724, 2725, 2726, 2734, 2735, 2736, 2737, 2738, 2743, 2744, 2745, 2746, 2747, 2748, 2749, 2750, 2751, 2752, 2753, 2754, 2755, 2756, 2757, 2758, 2759, 2760, 2761, 2762, 2763, 2764, 2765, 2766, 2767, 2768, 2769, 2770, 2771, 2772, 2773, 2774, 2775, 2776, 2777, 2778, 2779, 2780, 2781, 2782, 2783, 2784, 2785, 2786, 2787, 2788, 2789, 2790, 2791, 2792, 2793, 2794, 2795, 2796, 2797, 2798, 2799, 2800, 2801, 2802, 2805, 2806, 2807, 2808, 2809, 2810, 2811, 2812, 2813, 2814, 2815, 2816, 2817, 2818, 2819, 2820, 2821, 2822, 2823, 2824, 2825, 2826, 2827, 2829, 2830, 2831, 2832, 2833, 2834, 2835, 2836, 2837, 2838, 2839, 2840, 2845, 2846, 2847, 2848, 2849, 2850, 2851, 2852, 2853, 2854, 2855, 2856, 2857, 2858, 2859, 2860, 2861, 2862, 2863, 2864, 2865, 2866, 2867, 2868, 2869, 2870, 2871, 2872, 2873, 2874, 2875, 2876, 2877, 2878, 2879, 2880, 2881, 2882, 2883, 2884, 2885, 2886, 2887, 2888, 2889, 2890, 2891, 2892, 2893, 2894, 2895, 2896, 2897, 2898, 2899, 2900, 2901, 2902, 2903, 2904, 2906, 2907, 2908, 2909, 2910, 2911, 2912, 2913, 2914, 2915, 2916, 2917, 2918, 2919, 2920, 2921, 2922, 2923, 2924, 2925, 2926, 2927, 2928, 2930, 2931, 2932, 2933, 2934, 2935, 2936, 2937, 2938, 2939, 2940, 2941, 2942, 2946, 2947, 2948, 2949, 2950, 2951, 2952, 2953, 2954, 2955, 2956, 2957, 2958, 2959, 2960, 2961, 2962, 2963, 2964, 2965, 2966, 2967, 2968, 2969, 2970, 2971, 2972, 2973, 2974, 2975, 2976, 2977, 2978, 2979, 2980, 2981, 2982, 2983, 2984, 2985, 2986, 2987, 2988, 2989, 2990, 2991, 2992, 2993, 2994, 2995, 2996, 2997, 2998, 2999, 3000, 3001, 3002, 3003, 3004, 3005, 3006, 3007, 3008, 3009, 3010, 3011, 3012, 3013, 3014, 3015, 3016, 3017, 3018, 3019, 3020, 3021, 3022, 3023, 3024, 3025, 3026, 3027, 3028, 3029, 3032, 3033, 3034, 3035, 3036, 3037, 3038, 3039, 3040, 3041, 3042, 3043, 3044, 3045, 3046, 3047, 3048, 3049, 3050, 3051, 3052, 3053, 3054, 3055, 3056, 3057, 3058, 3059, 3060, 3061, 3062, 3063, 3064, 3065, 3066, 3067, 3068, 3069, 3070, 3071, 3072, 3073, 3074, 3075, 3076, 3077, 3078, 3079, 3080, 3081, 3082, 3083, 3084, 3085, 3086, 3087, 3088, 3089, 3090, 3091, 3092, 3093, 3094, 3095, 3096, 3097, 3098, 3099, 3100, 3101, 3102, 3103, 3104, 3105, 3106, 3107, 3108, 3109, 3110, 3111, 3112, 3113, 3114, 3115, 3116, 3117, 3118, 3119, 3120, 3121, 3122, 3123, 3124, 3125, 3126, 3127, 3128, 3129, 3139, 3140, 3141, 3142, 3143, 3144, 3145, 3146, 3147, 3148, 3149, 3150, 3151, 3152, 3153, 3154, 3155, 3156, 3157, 3158, 3159, 3160, 3161, 3162, 3163, 3164, 3165, 3166, 3167, 3168, 3169, 3170, 3171, 3172, 3173, 3174, 3175, 3176, 3177, 3178, 3179, 3180, 3181, 3182, 3183, 3184, 3185, 3186, 3187, 3188, 3189, 3190, 3191, 3192, 3193, 3194, 3195, 3196, 3197, 3198, 3199, 3200, 3201, 3202, 3203, 3204, 3205, 3206, 3207, 3208, 3209, 3210, 3211, 3212, 3213, 3214, 3215, 3216, 3217, 3218, 3219, 3220, 3221, 3222, 3223, 3224, 3225, 3226, 3228, 3229, 3230, 3240, 3241, 3242, 3243, 3244, 3245, 3246, 3247, 3248, 3249, 3250, 3251, 3252, 3253, 3254, 3255, 3256, 3257, 3258, 3259, 3260, 3261, 3262, 3263, 3264, 3265, 3266, 3267, 3268, 3269, 3270, 3271, 3272, 3273, 3274, 3275, 3276, 3277, 3278, 3279, 3280, 3281, 3282, 3283, 3284, 3285, 3286, 3287, 3288, 3289, 3290, 3291, 3292, 3293, 3294, 3295, 3296, 3297, 3298, 3299, 3300, 3301, 3302, 3303, 3304, 3305, 3306, 3307, 3308, 3309, 3310, 3311, 3312, 3313, 3314, 3315, 3316, 3317, 3318, 3319, 3320, 3321, 3322, 3323, 3324, 3325, 3326, 3341, 3342, 3343, 3344, 3345, 3346, 3347, 3348, 3349, 3350, 3351, 3352, 3353, 3354, 3355, 3356, 3357, 3358, 3359, 3360, 3361, 3362, 3363, 3364, 3365, 3366, 3367, 3368, 3369, 3370, 3371, 3372, 3373, 3374, 3375, 3376, 3377, 3378, 3379, 3380, 3381, 3382, 3383, 3384, 3385, 3386, 3387, 3388, 3389, 3390, 3391, 3392, 3393, 3394, 3395, 3396, 3397, 3398, 3399, 3400, 3401, 3402, 3403, 3404, 3405, 3406, 3407, 3408, 3409, 3410, 3411, 3412, 3413, 3414, 3415, 3416, 3417, 3418, 3419, 3420, 3421, 3422, 3423, 3424, 3425, 3426, 3427, 3442, 3443, 3444, 3445, 3446, 3447, 3448, 3449, 3450, 3451, 3452, 3453, 3454, 3455, 3456, 3457, 3458, 3459, 3460, 3461, 3462, 3463, 3464, 3465, 3466, 3467, 3468, 3469, 3470, 3471, 3472, 3473, 3474, 3475, 3476, 3477, 3478, 3479, 3480, 3481, 3482, 3483, 3484, 3485, 3486, 3487, 3488, 3489, 3490, 3491, 3492, 3493, 3494, 3495, 3496, 3497, 3498, 3499, 3500, 3501, 3502, 3503, 3504, 3505, 3506, 3507, 3508, 3509, 3510, 3511, 3512, 3513, 3514, 3515, 3516, 3517, 3518, 3519, 3520, 3521, 3522, 3523, 3524, 3525, 3526, 3527, 3528, 3529, 3543, 3544, 3545, 3546, 3547, 3548, 3549, 3550, 3551, 3552, 3553, 3554, 3555, 3556, 3557, 3558, 3559, 3560, 3561, 3562, 3563, 3564, 3565, 3566, 3567, 3568, 3569, 3570, 3571, 3572, 3573, 3574, 3575, 3576, 3577, 3578, 3579, 3580, 3581, 3582, 3583, 3584, 3585, 3586, 3587, 3588, 3589, 3590, 3591, 3592, 3593, 3594, 3595, 3596, 3597, 3598, 3599, 3600, 3601, 3602, 3603, 3604, 3605, 3606, 3607, 3608, 3609, 3610, 3611, 3612, 3613, 3614, 3615, 3616, 3617, 3618, 3619, 3620, 3621, 3622, 3623, 3624, 3625, 3626, 3627, 3628, 3629, 3630, 3631, 3644, 3645, 3646, 3647, 3648, 3649, 3650, 3651, 3652, 3653, 3654, 3655, 3656, 3657, 3658, 3659, 3660, 3661, 3662, 3663, 3664, 3665, 3666, 3667, 3668, 3669, 3670, 3671, 3672, 3673, 3674, 3675, 3676, 3677, 3678, 3679, 3680, 3681, 3682, 3683, 3684, 3685, 3686, 3687, 3688, 3689, 3690, 3691, 3692, 3693, 3694, 3695, 3696, 3697, 3698, 3699, 3700, 3701, 3702, 3703, 3704, 3705, 3706, 3707, 3708, 3709, 3710, 3711, 3712, 3713, 3714, 3715, 3716, 3717, 3718, 3719, 3720, 3721, 3722, 3723, 3724, 3725, 3726, 3727, 3728, 3729, 3730, 3731, 3745, 3746, 3747, 3748, 3749, 3750, 3751, 3752, 3753, 3754, 3755, 3756, 3757, 3758, 3759, 3760, 3761, 3762, 3763, 3764, 3765, 3766, 3767, 3768, 3769, 3770, 3771, 3772, 3773, 3774, 3775, 3776, 3777, 3778, 3779, 3780, 3781, 3782, 3783, 3784, 3785, 3786, 3787, 3788, 3789, 3790, 3791, 3792, 3793, 3794, 3795, 3796, 3797, 3798, 3799, 3800, 3801, 3802, 3803, 3804, 3805, 3806, 3807, 3808, 3809, 3810, 3811, 3812, 3813, 3814, 3815, 3816, 3817, 3818, 3819, 3820, 3821, 3822, 3823, 3824, 3825, 3826, 3827, 3828, 3829, 3830, 3831, 3832, 3846, 3847, 3848, 3849, 3850, 3851, 3852, 3853, 3854, 3855, 3856, 3857, 3858, 3859, 3860, 3861, 3862, 3863, 3864, 3865, 3866, 3867, 3868, 3869, 3870, 3871, 3872, 3873, 3874, 3875, 3876, 3877, 3878, 3879, 3880, 3881, 3882, 3883, 3884, 3885, 3886, 3887, 3888, 3889, 3890, 3891, 3892, 3893, 3894, 3895, 3896, 3897, 3898, 3899, 3900, 3901, 3902, 3903, 3904, 3905, 3906, 3907, 3908, 3909, 3910, 3911, 3912, 3913, 3914, 3915, 3916, 3917, 3918, 3919, 3920, 3921, 3922, 3923, 3924, 3925, 3926, 3927, 3928, 3929, 3930, 3931, 3932, 3948, 3949, 3950, 3951, 3952, 3953, 3954, 3955, 3956, 3957, 3958, 3959, 3960, 3961, 3962, 3963, 3964, 3965, 3966, 3967, 3968, 3969, 3970, 3971, 3972, 3973, 3974, 3975, 3976, 3977, 3978, 3979, 3980, 3981, 3982, 3983, 3984, 3985, 3986, 3987, 3988, 3989, 3990, 3991, 3992, 3993, 3994, 3995, 3996, 3997, 3998, 3999, 4000, 4001, 4002, 4003, 4004, 4005, 4006, 4007, 4008, 4009, 4010, 4011, 4012, 4013, 4014, 4015, 4016, 4017, 4018, 4019, 4020, 4021, 4022, 4023, 4024, 4025, 4026, 4027, 4028, 4029, 4030, 4031, 4032, 4033, 4051, 4052, 4053, 4054, 4055, 4056, 4057, 4058, 4059, 4060, 4061, 4062, 4063, 4064, 4065, 4066, 4067, 4068, 4069, 4070, 4071, 4072, 4073, 4074, 4075, 4076, 4077, 4078, 4079, 4080, 4081, 4082, 4083, 4084, 4085, 4086, 4087, 4088, 4089, 4090, 4091, 4092, 4093, 4094, 4095, 4096, 4097, 4098, 4099, 4100, 4101, 4102, 4103, 4104, 4105, 4106, 4107, 4108, 4109, 4110, 4111, 4112, 4113, 4114, 4115, 4116, 4117, 4118, 4119, 4120, 4121, 4122, 4123, 4124, 4125, 4126, 4127, 4128, 4129, 4130, 4131, 4132, 4133, 4134, 4154, 4155, 4156, 4157, 4158, 4159, 4160, 4161, 4162, 4163, 4164, 4165, 4166, 4167, 4168, 4169, 4170, 4171, 4172, 4173, 4174, 4175, 4176, 4177, 4178, 4179, 4180, 4181, 4182, 4183, 4184, 4185, 4186, 4187, 4188, 4189, 4190, 4191, 4192, 4193, 4194, 4195, 4196, 4197, 4198, 4199, 4200, 4201, 4202, 4203, 4204, 4205, 4206, 4207, 4208, 4209, 4210, 4211, 4212, 4213, 4214, 4215, 4216, 4217, 4218, 4219, 4220, 4221, 4222, 4223, 4224, 4225, 4226, 4227, 4228, 4229, 4230, 4231, 4232, 4233, 4260, 4261, 4262, 4263, 4264, 4265, 4266, 4267, 4268, 4269, 4270, 4271, 4272, 4273, 4274, 4275, 4276, 4277, 4278, 4279, 4280, 4281, 4282, 4283, 4284, 4285, 4286, 4287, 4288, 4289, 4290, 4291, 4292, 4293, 4294, 4295, 4296, 4297, 4298, 4299, 4300, 4301, 4302, 4303, 4304, 4305, 4306, 4307, 4308, 4309, 4310, 4311, 4312, 4313, 4314, 4315, 4316, 4317, 4318, 4319, 4320, 4321, 4322, 4323, 4324, 4325, 4326, 4327, 4328, 4329, 4330, 4331, 4332, 4333, 4363, 4364, 4365, 4366, 4367, 4368, 4369, 4370, 4376, 4377, 4378, 4379, 4380, 4381, 4382, 4383, 4384, 4385, 4386, 4387, 4388, 4389, 4390, 4391, 4392, 4393, 4394, 4395, 4396, 4397, 4398, 4399, 4400, 4401, 4402, 4403, 4404, 4405, 4406, 4407, 4408, 4409, 4410, 4411, 4412, 4413, 4414, 4415, 4416, 4417, 4418, 4419, 4420, 4421, 4422, 4423, 4424, 4425, 4426, 4427, 4428, 4429, 4430, 4463, 4464, 4465, 4466, 4467, 4468, 4469, 4470, 4471, 4477, 4478, 4479, 4480, 4481, 4482, 4483, 4484, 4485, 4486, 4487, 4488, 4489, 4490, 4491, 4492, 4493, 4494, 4495, 4496, 4497, 4498, 4499, 4500, 4501, 4502, 4503, 4504, 4505, 4506, 4507, 4508, 4509, 4510, 4511, 4512, 4513, 4514, 4515, 4516, 4517, 4518, 4519, 4520, 4521, 4522, 4523, 4524, 4525, 4526, 4527, 4528, 4529, 4530, 4531, 4564, 4565, 4566, 4567, 4568, 4569, 4570, 4571, 4577, 4581, 4582, 4583, 4584, 4585, 4586, 4587, 4588, 4589, 4590, 4591, 4592, 4593, 4594, 4595, 4596, 4597, 4598, 4599, 4600, 4601, 4602, 4603, 4604, 4605, 4606, 4607, 4608, 4609, 4610, 4611, 4612, 4613, 4614, 4615, 4616, 4617, 4618, 4619, 4620, 4621, 4622, 4623, 4624, 4625, 4626, 4629, 4630, 4631, 4632, 4633, 4664, 4665, 4666, 4667, 4668, 4669, 4670, 4671, 4672, 4678, 4682, 4683, 4684, 4685, 4686, 4687, 4688, 4689, 4690, 4691, 4692, 4693, 4694, 4695, 4696, 4697, 4698, 4699, 4700, 4702, 4703, 4704, 4705, 4706, 4707, 4708, 4709, 4710, 4711, 4712, 4713, 4714, 4715, 4716, 4717, 4718, 4719, 4720, 4722, 4723, 4724, 4725, 4726, 4727, 4730, 4731, 4732, 4733, 4734, 4765, 4766, 4767, 4768, 4769, 4770, 4771, 4772, 4773, 4774, 4775, 4778, 4779, 4780, 4781, 4782, 4783, 4784, 4785, 4786, 4787, 4788, 4789, 4790, 4791, 4792, 4793, 4794, 4795, 4796, 4797, 4798, 4799, 4803, 4804, 4805, 4806, 4807, 4808, 4809, 4810, 4811, 4812, 4813, 4814, 4815, 4816, 4817, 4818, 4819, 4820, 4821, 4826, 4827, 4828, 4829, 4831, 4832, 4833, 4865, 4866, 4867, 4868, 4869, 4870, 4871, 4872, 4873, 4874, 4875, 4876, 4879, 4880, 4881, 4882, 4883, 4884, 4885, 4886, 4887, 4888, 4889, 4890, 4891, 4892, 4893, 4894, 4895, 4896, 4897, 4898, 4899, 4900, 4905, 4906, 4907, 4908, 4909, 4910, 4911, 4912, 4913, 4914, 4915, 4916, 4917, 4918, 4919, 4920, 4921, 4966, 4967, 4968, 4969, 4970, 4971, 4972, 4973, 4974, 4975, 4976, 4977, 4978, 4979, 4980, 4981, 4982, 4983, 4984, 4985, 4986, 4987, 4988, 4989, 4990, 4991, 4992, 4993, 4994, 4995, 4996, 4997, 4998, 4999, 5000, 5001, 5007, 5008, 5009, 5010, 5011, 5012, 5013, 5014, 5015, 5016, 5017, 5018, 5019, 5020, 5021, 5067, 5068, 5069, 5070, 5071, 5072, 5073, 5074, 5075, 5076, 5077, 5078, 5079, 5080, 5081, 5082, 5083, 5084, 5085, 5086, 5087, 5088, 5089, 5090, 5091, 5092, 5093, 5097, 5098, 5099, 5100, 5101, 5112, 5113, 5114, 5115, 5116, 5117, 5118, 5119, 5169, 5170, 5171, 5172, 5173, 5174, 5175, 5176, 5177, 5178, 5179, 5180, 5181, 5182, 5183, 5184, 5185, 5186, 5187, 5188, 5189, 5190, 5191, 5192, 5193, 5200, 5201, 5202, 5214, 5215, 5216, 5217, 5218, 5271, 5272, 5273, 5274, 5275, 5276, 5277, 5278, 5279, 5280, 5281, 5282, 5283, 5284, 5285, 5286, 5287, 5288, 5289, 5290, 5291, 5293, 5316, 5317, 5318, 5319, 5373, 5374, 5375, 5376, 5377, 5378, 5379, 5380, 5381, 5382, 5383, 5384, 5385, 5386, 5387, 5388, 5389, 5390, 5418, 5419, 5420, 5476, 5477, 5478, 5479, 5480, 5481, 5482, 5483, 5484, 5485, 5486, 5487, 5488, 5489, 5490, 5491, 5519, 5520, 5579, 5580, 5581, 5582, 5583, 5584, 5585, 5586, 5587, 5588, 5589, 5590, 5591, 5682, 5683, 5684, 5685, 5686, 5687, 5688, 5689, 5690, 5691, 5692, 5693, 5784, 5785, 5786, 5787, 5788, 5791]

In [ ]:
%%script true
# The result is at the next cell.
# Start the optimisation

optimised_ind = optimisation(turbine_loc.shape[0], rows, cols, _wind_data, gene_space)[0].tolist()
optimised_ind.sort()
print(optimised_ind)

In [ ]:
# %%script true
# The indices of the optimised turbines with random seed 0 (Generation: 1780 (grounded to 10); time cost: 376.0 sec; fitness value:2645)
optimised_ind = [19, 22, 129, 225, 237, 331, 351, 436, 534, 552, 651, 746, 774, 842, 869, 949, 959, 1040, 1048, 1063, 1075, 1145, 1152, 1162, 1180, 1249, 1283, 1355, 1383, 1463, 1476, 1544, 1567, 1584, 1656, 1684, 1755, 1768, 1781, 1785, 1851, 1881, 1891, 1945, 1969, 1984, 1995, 2043, 2062, 2072, 2090, 2154, 2185, 2208, 2246, 2257, 2280, 2353, 2355, 2364, 2396, 2407, 2413, 2444, 2496, 2510, 2551, 2564, 2569, 2614, 2625, 2637, 2645, 2657, 2668, 2690, 2712, 2724, 2737, 2754, 2783, 2793, 2801, 2806, 2819, 2829, 2846, 2851, 2866, 2880, 2909, 2935, 2949, 2955, 2969, 2979, 2991, 3002, 3007, 3026, 3034, 3039, 3044, 3047, 3059, 3073, 3088, 3106, 3123, 3129, 3165, 3196, 3216, 3244, 3273, 3299, 3312, 3349, 3354, 3373, 3408, 3425, 3443, 3462, 3490, 3494, 3505, 3523, 3552, 3568, 3597, 3616, 3627, 3667, 3683, 3715, 3724, 3732, 3748, 3764, 3778, 3806, 3822, 3828, 3846, 3857, 3863, 3893, 3906, 3927, 3951, 3969, 4014, 4025, 4057, 4072, 4099, 4109, 4129, 4165, 4179, 4202, 4217, 4224, 4277, 4294, 4300, 4307, 4329, 4364, 4393, 4406, 4427, 4497, 4500, 4519, 4525, 4588, 4605, 4616, 4664, 4671, 4687, 4695, 4700, 4726, 4774, 4781, 4819, 4870, 4895, 4915, 4968, 4983, 4994, 5000, 5014, 5020, 5077, 5086, 5114, 5171, 5180, 5190, 5191, 5275, 5279, 5319, 5373, 5390, 5489, 5580, 5686, 5688, 5787]


In [ ]:
# The optimised turbine locations
optimised_loc = conv.gene_to_pos(optimised_ind)
optimised_loc = np.array(optimised_loc)

Then we visualise the results.

In [ ]:
# Create a folium map centered around the site
m = folium.Map(location=[55.6741, -4.2713], zoom_start=12, tiles='CartoDB.PositronNoLabels')

# Remove the filling display of the geojson file but keep the boundary
folium.GeoJson(
    boundary,
    style_function=lambda x: {
        'fillOpacity': 0.1,  # Make the fill transparent
        'fillColor': '#00FF00',
        'color': '#005000',  # Keep the boundary visible
        'weight': 2,
    }
).add_to(m)

# Add the optimised wind turbines to the map
for i in range(optimised_loc.shape[0]):  # add turbine locations
    folium.Marker(
        location=[optimised_loc[i, 0], optimised_loc[i, 1]],
        icon=folium.CustomIcon(icon_image=image_file, icon_size=(20, 20), icon_anchor=(10, 20)),
        color='red'
    ).add_to(m)

# Display land cover type within the geojson boundary
land_cover_file = 'Land/raw/C3S-LC-L4-LCCS-Map-300m-P1Y-2022-v2.1.1.nc'
with netCDF4.Dataset(land_cover_file, 'r') as file:
    land_class = file.variables['lccs_class']
    lat = file.variables['lat'][:]
    lon = file.variables['lon'][:]

    # Filter points within the bounding box
    y_range = np.argwhere((lat < bounds[3]) & (lat > bounds[1])).flatten()
    x_range = np.argwhere((lon < bounds[2]) & (lon > bounds[0])).flatten()

    # Extract the land cover data within the bounding box
    land_cover = land_class[0, y_range.min():y_range.max()+1, x_range.min():x_range.max()+1].data

    # Create an RGBA image for the land cover data
    rgba_img = np.zeros((len(y_range), len(x_range), 4), dtype=np.uint8)
    colourmap = {
        70: [0, 60, 0, 128],
        100: [140, 160, 0, 128],
        110: [190, 150, 0, 128],
        130: [255, 180, 50, 128],
        180: [0, 220, 130, 128],
        210: [0, 70, 200, 128],
    }

    for key, color in colourmap.items():
        rgba_img[land_cover == key] = color

    # Assign transparent color to data outside the geojson boundary
    for i, lat_val in enumerate(lat[y_range]):
        for j, lon_val in enumerate(lon[x_range]):
            point = Point(lon_val, lat_val)
            if not boundary.union_all().contains(point):
                rgba_img[i, j, 3] = 0  # Set alpha to 0 for transparency

    # Add the RGBA image to the map
    # folium.raster_layers.ImageOverlay(
    #     image=rgba_img,
    #     bounds=bounding_box,
    #     origin='upper',
    #     opacity=1
    # ).add_to(m)

# Add a legend to the map
legend_html = '''
    <div style="position: fixed; 
                bottom: 50px; right: 50px; width: auto; height: auto; 
                background-color: rgba(255, 255, 255, 0.8); border: 2px solid black; z-index:9999; font-size:14px;">
        <h4 style="margin: 10px; font-weight: bold; font-size: 20px;">Legend</h4>
        <ul style="list-style: none; padding: 0; margin: 10px;">
        <li><img src="https://github.com/ShitianZhang22/Wind-Farm-Layout-Optimisation/blob/main/icon/turbine-small.png?raw=true" style="width: 15px; height: 15px; margin-right: 2px;">Wind Turbine</li>
            <li><span style="background-color: black; width: 12px; height: 3px; display: inline-block; margin-right: 5px; margin-bottom: 4px;"></span>Site Boundary</li>
            <li><span style="background-color: rgba(0, 60, 0, 0.5); width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></span>Tree Cover, Needleleaved, Evergreen, Closed to Open</li>
            <li><span style="background-color: rgba(140, 160, 0, 0.5); width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></span>Mosaic Tree and Shrub (>50%) / Herbaceous Cover (<50%)</li>
            <li><span style="background-color: rgba(190, 150, 0, 0.5); width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></span>Mosaic Herbaceous cover (>50%) / Tree and Shrub (<50%)</li>
            <li><span style="background-color: rgba(255, 180, 50, 0.5); width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></span>Grassland</li>
            <li><span style="background-color: rgba(0, 220, 130, 0.5); width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></span>Shrub or Herbaceous Cover, Flooded, Fresh/Saline/Brackish Water</li>
            <li><span style="background-color: rgba(0, 70, 200, 0.5); width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></span>Water Bodies</li>
        </ul>
    </div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Display the map
m